# Train a language model, end to end

This notebook walks the whole text pipeline in-process: **tokenizer → corpus →
pretraining → instruction tuning → evaluation → generation**. It uses the small
corpus bundled inside the package, so it runs offline on CPU in a couple of
minutes — the point is to see every moving part, not to produce a good model.

The CLI equivalent of everything here is `minimodel quickstart`; real
workflows live in `docs/recipes.md`.

In [ ]:
from pathlib import Path

import torch

from minimodel.datasets.builtin import builtin_records
from minimodel.tokenization import BPETokenizer

work = Path("../runs/notebook-lm")
work.mkdir(parents=True, exist_ok=True)

# 1. Train a byte-level BPE tokenizer on the bundled corpus.
texts = [record["text"] for record in builtin_records("pretrain", repeat=8)]
tokenizer = BPETokenizer.train(texts, vocab_size=1024)
tokenizer.save(work / "tokenizer.json")

print(tokenizer)
print("bytes/token:", round(tokenizer.compression_ratio(texts[:32]), 2))
print(tokenizer.encode("The river runs east."))

In [ ]:
# 2. Tokenize into memmapped shards and build datasets.
from minimodel.datasets import PackedTextDataset, tokenize_text_records

stats = tokenize_text_records(({"text": t} for t in texts), tokenizer, work / "corpus")
print(f"{stats['n_tokens']:,} tokens from {stats['n_documents']} documents")

train_dataset = PackedTextDataset(work / "corpus", seq_len=64, seed=0)
val_dataset = PackedTextDataset(work / "corpus", seq_len=64, seed=999)

inputs, labels = train_dataset[0]
print("window:", inputs.shape, "| labels are inputs shifted:", bool((inputs[1:] == labels[:-1]).all()))

In [ ]:
# 3. Build a model. Templates carry verified parameter counts; overrides make
#    this one tiny enough for a notebook.
from minimodel.architectures import build_model

model = build_model(
    "dense_3m",
    overrides={
        "vocab_size": tokenizer.vocab_size,
        "dim": 128, "n_layers": 4, "n_heads": 4, "head_dim": 32, "n_kv_heads": 2,
        "ffn_hidden": 352, "max_seq_len": 128, "window": 64,
    },
    verify_budget=False,
)
print(f"{model.num_parameters():,} parameters")
model.parameter_breakdown()

In [ ]:
# 4. Pretrain. Loss should start near ln(vocab) = 6.93 and fall fast.
from minimodel.training import Trainer, TrainerConfig

config = TrainerConfig(
    run_name="pretrain", output_dir=str(work), max_steps=300,
    batch_size=8, seq_len=64, lr=3e-3, warmup=0.1,
    log_every=50, eval_every=100, eval_batches=4, save_every=0, resume=False,
)
result = Trainer(model, config, train_dataset=train_dataset,
                 eval_dataset=val_dataset, tokenizer=tokenizer).fit()
print(f"final loss {result.final_loss:.3f} → perplexity {result.final_perplexity:.1f}")

In [ ]:
# 5. Plot the curve (falls back to ASCII without matplotlib).
from minimodel.checkpointing import plot_loss_curve, summarize_run

output = plot_loss_curve(work / "pretrain", work / "loss.png")
print(output)
summarize_run(work / "pretrain" / "metrics.jsonl")

In [ ]:
# 6. Instruction-tune. The supervised corpus carries a label mask, so only
#    assistant tokens contribute loss — watch supervised_frac in the logs.
from minimodel.datasets import SupervisedDataset, tokenize_chat_records
from minimodel.training import InstructTrainer, InstructTrainerConfig

tokenize_chat_records(builtin_records("sft", repeat=8), tokenizer, work / "sft")
sft_config = InstructTrainerConfig(
    run_name="sft", output_dir=str(work), max_steps=100,
    batch_size=8, seq_len=64, lr=5e-4, log_every=25,
    eval_every=0, save_every=0, resume=False,
)
sft_result = InstructTrainer(
    model, sft_config,
    train_dataset=SupervisedDataset(work / "sft", seq_len=64),
    tokenizer=tokenizer,
).fit()
print(f"SFT final loss {sft_result.final_loss:.3f}")

In [ ]:
# 7. Evaluate on the bundled offline tasks (likelihood-scored), then sample.
from minimodel.benchmarking import run_suite
from minimodel.inference import SamplingConfig, generate_text

bench = run_suite(model, tokenizer, perplexity_corpus=work / "corpus",
                  include_throughput=False, model_name="notebook-lm")
print(bench.headline())

for temperature in (0.0, 0.8):
    text = generate_text(model, tokenizer, "The river runs",
                         max_new_tokens=40, temperature=temperature,
                         repetition_penalty=1.1, seed=0)
    print(f"\nT={temperature}: {text}")

## Where to go next

- Scale the same code up: swap the overrides for a real template
  (`build_model("dense_12m")`), point `PackedTextDataset` at a tokenized
  TinyStories or FineWeb-Edu corpus, raise `max_steps` — nothing else changes.
- `docs/recipes.md` has the CLI versions of the full lifecycle including DPO,
  RLVR and merging.
- Notebook 02 does the same tour for the image side (PixelGPT + diffusion).